<img src="MUML_header.svg" alt="Header" style="width:100%; height:auto;">

### Implementing a Recurrent Neural Network from scratch for next-character prediction

In this exercise, you will implement a simple recurrent neural network (RNN) from scratch using PyTorch, with the goal of predicting the next character in a text sequence. Unlike the previous practice, where you used a multilayer perceptron (MLP), here you will build a model that explicitly handles sequential dependencies through a recurrent hidden state.

The purpose of this practice is not only to train a working model but also to understand deeply how an RNN is constructed, what shapes its parameters must have, how the hidden state evolves, and how mini-batch training works.

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import math
import random

In [2]:
# Reproducibility
torch.manual_seed(42)
random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


Just like in the first practice, you need to transform text data into numerical form. Start by creating a vocabulary dictionary that maps each character to an integer index. Then, create a custom Dataset class that, given a long string of text, returns input sequences of a fixed length (context_size) and their corresponding target characters (the next character to predict).

This dataset will also be responsible for turning input indices into embeddings, since in this assignment we will use a trainable embedding layer to represent characters.

In [3]:
with open("PrideAndPrejudicePreface.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("Number of characters:", len(text))
print(text[:500])


Number of characters: 24937
﻿Walt Whitman has somewhere a fine and just distinction between “loving
by allowance” and “loving with personal love.” This distinction applies
to books as well as to men and women; and in the case of the not very
numerous authors who are the objects of the personal affection, it
brings a curious consequence with it. There is much more difference as
to their best work than in the case of those others who are loved “by
allowance” by convention, and because it is felt to be the right and
proper th


In [4]:
# Vocabulary
chars = sorted(set(text))
vocab_size = len(chars)

char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}

# Encode the text as integers
encoded_text = torch.tensor(
    [char_to_idx[ch] for ch in text],
    dtype=torch.long
)

print("Vocabulary size:", vocab_size)
print("Encoded text shape:", encoded_text.shape)

Vocabulary size: 70
Encoded text shape: torch.Size([24937])


In [5]:
class CharDataset(Dataset):
    def __init__(self, encoded_text, context_size):
        self.data = encoded_text
        self.context_size = context_size

    def __len__(self):
        return len(self.data) - self.context_size

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.context_size]
        y = self.data[idx + 1 : idx + self.context_size + 1]
        return x, y

In [6]:
context_size = 40
batch_size = 64

dataset = CharDataset(encoded_text, context_size)

loader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True
)

x_batch, y_batch = next(iter(loader))

print("Input batch shape:", x_batch.shape)
print("Target batch shape:", y_batch.shape)

Input batch shape: torch.Size([64, 40])
Target batch shape: torch.Size([64, 40])


Now you will implement the core building block: the RNN cell. This cell takes two inputs at each time step:
* The current input vector (a character embedding)
* The previous hidden state

It then produces:
* The new hidden state (which will be used at the next time step)
* An output (used to predict the next character)

The equations you have to use in the RNN are:
* $H_t=\phi(X_tW_{xh}+H_{t-1}W_{hh}+b_h)$
* $O_t=H_tW_{hq}+b_q$

Your implementation will need to define the weight matrices (`W_xh`, `W_hh`, `W_hy`) and biases. Pay close attention to their shapes:

* `W_xh`: (input_size, hidden_size)
* `W_hh`: (hidden_size, hidden_size)
* `W_hy`: (hidden_size, vocab_size)

The forward pass should iterate over the time steps of the sequence, updating the hidden state and generating outputs.

In [7]:
class RNNCell(nn.Module):
    def __init__(self, input_size, hidden_size, vocab_size):
        super().__init__()

        self.W_xh = nn.Parameter(
            torch.randn(input_size, hidden_size) / math.sqrt(input_size)
        )
        self.W_hh = nn.Parameter(
            torch.randn(hidden_size, hidden_size) / math.sqrt(hidden_size)
        )
        self.b_h = nn.Parameter(torch.zeros(hidden_size))

        self.W_hy = nn.Parameter(
            torch.randn(hidden_size, vocab_size) / math.sqrt(hidden_size)
        )
        self.b_y = nn.Parameter(torch.zeros(vocab_size))

    def forward(self, x_t, h_prev):
        # x_t: (batch_size, input_size)
        # h_prev: (batch_size, hidden_size)

        h_t = torch.tanh(
            x_t @ self.W_xh
            + h_prev @ self.W_hh
            + self.b_h
        )

        logits = h_t @ self.W_hy + self.b_y

        return h_t, logits

Wrap your RNN cell into a PyTorch `nn.Module` class. The model should:
* Receive a batch of sequences (indices of characters)
* Pass them through an `nn.Embedding` layer to get dense vector representations.
* Use the RNN cell to process the sequence step by step
* Return the predictions for each time step (so that you can compare them to the true next characters)

Remember: the hidden state must have shape (`batch_size`, `hidden_size`) because each sequence in the batch needs its own memory.

In [8]:
class CharRNN(nn.Module):
    def __init__(self, vocab_size, embedding_size, hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_size)
        self.rnn_cell = RNNCell(
            input_size=embedding_size,
            hidden_size=hidden_size,
            vocab_size=vocab_size
        )

        self.hidden_size = hidden_size

    def forward(self, x):
        # x: (batch_size, sequence_length)
        batch_size, seq_len = x.shape

        embeddings = self.embedding(x)
        # embeddings: (batch_size, seq_len, embedding_size)

        h = torch.zeros(
            batch_size,
            self.hidden_size,
            device=x.device
        )

        outputs = []

        for t in range(seq_len):
            x_t = embeddings[:, t, :]
            h, logits = self.rnn_cell(x_t, h)
            outputs.append(logits)

        # (batch_size, seq_len, vocab_size)
        return torch.stack(outputs, dim=1)

In [9]:
embedding_size = 32
hidden_size = 128

model = CharRNN(
    vocab_size=vocab_size,
    embedding_size=embedding_size,
    hidden_size=hidden_size
).to(device)

x_batch, y_batch = next(iter(loader))
x_batch = x_batch.to(device)

with torch.no_grad():
    logits = model(x_batch)

print("Input:", x_batch.shape)
print("Output:", logits.shape)

assert logits.shape == (
    x_batch.shape[0],
    x_batch.shape[1],
    vocab_size
)

Input: torch.Size([64, 40])
Output: torch.Size([64, 40, 70])


Now implement a training loop. At each iteration:
* Get a mini-batch of sequences from the dataset (fix the length of the sequences (`context_size`))
* Initialize the hidden state to zeros
* Perform a forward pass through the model
* Compute the loss (e.g., cross-entropy) between predictions and true next characters
* Backpropagate the gradients and update the weights with an optimizer (use `nn.utils.clip_grad_norm_`

In [10]:
def train_model(
    model,
    loader,
    epochs=20,
    learning_rate=0.003,
    clip_value=1.0
):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate
    )

    history = []

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        total_chars = 0

        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            logits = model(x)

            # (batch_size * seq_len, vocab_size)
            logits = logits.reshape(-1, vocab_size)

            # (batch_size * seq_len,)
            targets = y.reshape(-1)

            loss = criterion(logits, targets)

            loss.backward()

            # Prevent excessively large gradients
            nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=clip_value
            )

            optimizer.step()

            num_chars = targets.numel()
            total_loss += loss.item() * num_chars
            total_chars += num_chars

        epoch_loss = total_loss / total_chars
        history.append(epoch_loss)

        print(
            f"Epoch {epoch + 1:3d}/{epochs} | "
            f"Loss: {epoch_loss:.4f} | "
            f"Perplexity: {math.exp(min(epoch_loss, 20)):.2f}"
        )

    return history

In [11]:
history = train_model(
    model,
    loader,
    epochs=20,
    learning_rate=0.003,
    clip_value=1.0
)

Epoch   1/20 | Loss: 1.9406 | Perplexity: 6.96
Epoch   2/20 | Loss: 1.3954 | Perplexity: 4.04
Epoch   3/20 | Loss: 1.2052 | Perplexity: 3.34
Epoch   4/20 | Loss: 1.1087 | Perplexity: 3.03
Epoch   5/20 | Loss: 1.0491 | Perplexity: 2.86
Epoch   6/20 | Loss: 1.0106 | Perplexity: 2.75
Epoch   7/20 | Loss: 0.9809 | Perplexity: 2.67
Epoch   8/20 | Loss: 0.9611 | Perplexity: 2.61
Epoch   9/20 | Loss: 0.9412 | Perplexity: 2.56
Epoch  10/20 | Loss: 0.9278 | Perplexity: 2.53
Epoch  11/20 | Loss: 0.9143 | Perplexity: 2.50
Epoch  12/20 | Loss: 0.9059 | Perplexity: 2.47
Epoch  13/20 | Loss: 0.8970 | Perplexity: 2.45
Epoch  14/20 | Loss: 0.8885 | Perplexity: 2.43
Epoch  15/20 | Loss: 0.8803 | Perplexity: 2.41
Epoch  16/20 | Loss: 0.8744 | Perplexity: 2.40
Epoch  17/20 | Loss: 0.8693 | Perplexity: 2.39
Epoch  18/20 | Loss: 0.8634 | Perplexity: 2.37
Epoch  19/20 | Loss: 0.8596 | Perplexity: 2.36
Epoch  20/20 | Loss: 0.8540 | Perplexity: 2.35


Finally, implement a generate function that starts with a seed sequence of characters. Iteratively feeds the predicted character back into the model and produces new text character by character.

In [12]:
@torch.no_grad()
def generate(model, seed, max_new_chars=300, temperature=0.8):
    if temperature <= 0:
        raise ValueError("Temperature must be greater than zero.")

    if not seed:
        raise ValueError("Seed must not be empty.")

    unknown_chars = [ch for ch in seed if ch not in char_to_idx]
    if unknown_chars:
        raise ValueError(f"Characters not in vocabulary: {unknown_chars}")

    model.eval()

    context = torch.tensor(
        [[char_to_idx[ch] for ch in seed]],
        dtype=torch.long,
        device=device
    )

    generated = list(seed)

    for _ in range(max_new_chars):
        # Keep only the most recent context_size characters
        context = context[:, -context_size:]

        logits = model(context)
        next_logits = logits[:, -1, :] / temperature

        probabilities = torch.softmax(next_logits, dim=-1)

        next_idx = torch.multinomial(
            probabilities,
            num_samples=1
        ).item()

        generated.append(idx_to_char[next_idx])

        next_tensor = torch.tensor(
            [[next_idx]],
            dtype=torch.long,
            device=device
        )

        context = torch.cat([context, next_tensor], dim=1)

    return "".join(generated)

In [13]:
print(generate(
    model,
    seed=text[:10],
    max_new_chars=300,
    temperature=0.8
))

﻿Walt Whit Pride--iffecenessank or by the other limity”) fact are she did not at least given paraim
to Lucas (why
nature when Mr. Spechaps, about hid so slainst wert to still not and men and was so gerious indeed a ded have for Austenialled nective-ysh in of course betap of out of this does not in she edryo a


Compare how the model performs with different hidden sizes, context sizes or training epochs.

In [14]:
def run_experiment(
    hidden_size=128,
    context_size=40,
    epochs=20,
    embedding_size=32,
    batch_size=64,
    learning_rate=0.003
):
    dataset = CharDataset(encoded_text, context_size)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True
    )

    experiment_model = CharRNN(
        vocab_size=vocab_size,
        embedding_size=embedding_size,
        hidden_size=hidden_size
    ).to(device)

    history = train_model(
        experiment_model,
        loader,
        epochs=epochs,
        learning_rate=learning_rate
    )

    return {
        "hidden_size": hidden_size,
        "context_size": context_size,
        "epochs": epochs,
        "final_loss": history[-1],
        "model": experiment_model,
        "history": history
    }

In [15]:
hidden_sizes = [64, 128, 256]
hidden_results = []

for size in hidden_sizes:
    result = run_experiment(
        hidden_size=size,
        context_size=40,
        epochs=20
    )

    hidden_results.append(result)

for result in hidden_results:
    print(
        f"Hidden size: {result['hidden_size']:3d} | "
        f"Final loss: {result['final_loss']:.4f}"
    )

Epoch   1/20 | Loss: 2.2042 | Perplexity: 9.06
Epoch   2/20 | Loss: 1.7697 | Perplexity: 5.87
Epoch   3/20 | Loss: 1.6491 | Perplexity: 5.20
Epoch   4/20 | Loss: 1.5858 | Perplexity: 4.88
Epoch   5/20 | Loss: 1.5459 | Perplexity: 4.69
Epoch   6/20 | Loss: 1.5188 | Perplexity: 4.57
Epoch   7/20 | Loss: 1.4989 | Perplexity: 4.48
Epoch   8/20 | Loss: 1.4840 | Perplexity: 4.41
Epoch   9/20 | Loss: 1.4732 | Perplexity: 4.36
Epoch  10/20 | Loss: 1.4648 | Perplexity: 4.33
Epoch  11/20 | Loss: 1.4573 | Perplexity: 4.29
Epoch  12/20 | Loss: 1.4514 | Perplexity: 4.27
Epoch  13/20 | Loss: 1.4459 | Perplexity: 4.25
Epoch  14/20 | Loss: 1.4410 | Perplexity: 4.23
Epoch  15/20 | Loss: 1.4383 | Perplexity: 4.21
Epoch  16/20 | Loss: 1.4336 | Perplexity: 4.19
Epoch  17/20 | Loss: 1.4306 | Perplexity: 4.18
Epoch  18/20 | Loss: 1.4290 | Perplexity: 4.17
Epoch  19/20 | Loss: 1.4262 | Perplexity: 4.16
Epoch  20/20 | Loss: 1.4238 | Perplexity: 4.15
Epoch   1/20 | Loss: 1.9583 | Perplexity: 7.09
Epoch   2/20 

In [16]:
context_sizes = [20, 40, 80]
context_results = []

for size in context_sizes:
    result = run_experiment(
        hidden_size=128,
        context_size=size,
        epochs=20
    )

    context_results.append(result)

for result in context_results:
    print(
        f"Context size: {result['context_size']:3d} | "
        f"Final loss: {result['final_loss']:.4f}"
    )

Epoch   1/20 | Loss: 2.0559 | Perplexity: 7.81
Epoch   2/20 | Loss: 1.5999 | Perplexity: 4.95
Epoch   3/20 | Loss: 1.4431 | Perplexity: 4.23
Epoch   4/20 | Loss: 1.3539 | Perplexity: 3.87
Epoch   5/20 | Loss: 1.2956 | Perplexity: 3.65
Epoch   6/20 | Loss: 1.2534 | Perplexity: 3.50
Epoch   7/20 | Loss: 1.2213 | Perplexity: 3.39
Epoch   8/20 | Loss: 1.1975 | Perplexity: 3.31
Epoch   9/20 | Loss: 1.1783 | Perplexity: 3.25
Epoch  10/20 | Loss: 1.1612 | Perplexity: 3.19
Epoch  11/20 | Loss: 1.1487 | Perplexity: 3.15
Epoch  12/20 | Loss: 1.1374 | Perplexity: 3.12
Epoch  13/20 | Loss: 1.1280 | Perplexity: 3.09
Epoch  14/20 | Loss: 1.1202 | Perplexity: 3.07
Epoch  15/20 | Loss: 1.1120 | Perplexity: 3.04
Epoch  16/20 | Loss: 1.1049 | Perplexity: 3.02
Epoch  17/20 | Loss: 1.0980 | Perplexity: 3.00
Epoch  18/20 | Loss: 1.0948 | Perplexity: 2.99
Epoch  19/20 | Loss: 1.0887 | Perplexity: 2.97
Epoch  20/20 | Loss: 1.0838 | Perplexity: 2.96
Epoch   1/20 | Loss: 1.9274 | Perplexity: 6.87
Epoch   2/20 

In [17]:
epoch_counts = [5, 20, 50]
epoch_results = []

for n_epochs in epoch_counts:
    result = run_experiment(
        hidden_size=128,
        context_size=40,
        epochs=n_epochs
    )

    epoch_results.append(result)

for result in epoch_results:
    print(
        f"Epochs: {result['epochs']:3d} | "
        f"Final loss: {result['final_loss']:.4f}"
    )

Epoch   1/5 | Loss: 1.9533 | Perplexity: 7.05
Epoch   2/5 | Loss: 1.4067 | Perplexity: 4.08
Epoch   3/5 | Loss: 1.2137 | Perplexity: 3.37
Epoch   4/5 | Loss: 1.1159 | Perplexity: 3.05
Epoch   5/5 | Loss: 1.0584 | Perplexity: 2.88
Epoch   1/20 | Loss: 1.9395 | Perplexity: 6.96
Epoch   2/20 | Loss: 1.3908 | Perplexity: 4.02
Epoch   3/20 | Loss: 1.2006 | Perplexity: 3.32
Epoch   4/20 | Loss: 1.1022 | Perplexity: 3.01
Epoch   5/20 | Loss: 1.0455 | Perplexity: 2.84
Epoch   6/20 | Loss: 1.0062 | Perplexity: 2.74
Epoch   7/20 | Loss: 0.9769 | Perplexity: 2.66
Epoch   8/20 | Loss: 0.9565 | Perplexity: 2.60
Epoch   9/20 | Loss: 0.9407 | Perplexity: 2.56
Epoch  10/20 | Loss: 0.9250 | Perplexity: 2.52
Epoch  11/20 | Loss: 0.9129 | Perplexity: 2.49
Epoch  12/20 | Loss: 0.9032 | Perplexity: 2.47
Epoch  13/20 | Loss: 0.8958 | Perplexity: 2.45
Epoch  14/20 | Loss: 0.8889 | Perplexity: 2.43
Epoch  15/20 | Loss: 0.8796 | Perplexity: 2.41
Epoch  16/20 | Loss: 0.8725 | Perplexity: 2.39
Epoch  17/20 | Los

In [18]:
import pandas as pd

all_results = (
    hidden_results
    + context_results
    + epoch_results
)

results_df = pd.DataFrame([
    {
        "hidden_size": r["hidden_size"],
        "context_size": r["context_size"],
        "epochs": r["epochs"],
        "final_loss": r["final_loss"]
    }
    for r in all_results
])

results_df

,hidden_size,context_size,epochs,final_loss
0,64,40,20,1.423817
1,128,40,20,0.862792
2,256,40,20,0.381429
3,128,20,20,1.083792
4,128,40,20,0.844181
5,128,80,20,0.704841
6,128,40,5,1.058365
7,128,40,20,0.854750
8,128,40,50,0.800948
